# Workers AI decision models with LangChain

`CloudflareWorkersAIDecisionModel` calls Clef or Clef Flash with a state and typed questions. It returns probabilities rather than a chat message. This notebook shows text and vision decisions through the library's production REST path.

Run `make notebook_sync` and `make notebook_lab` from the repository root. Set `CF_ACCOUNT_ID` and `CF_AI_API_TOKEN` in the repo-root `.env`.

[Clef model documentation](https://developers.cloudflare.com/workers-ai/models/clef/)

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain_cloudflare import CloudflareWorkersAIDecisionModel

load_dotenv(".env")
account_id = os.getenv("CF_ACCOUNT_ID")
api_token = os.getenv("CF_AI_API_TOKEN")
missing = [
    name
    for name, value in {
        "CF_ACCOUNT_ID": account_id,
        "CF_AI_API_TOKEN": api_token,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(
        f"Set these variables before running the notebook: {', '.join(missing)}"
    )

## Text and structured questions

`noul` asks a yes/no question, `choice` selects from named options, and `score` uses an ordered rubric. The keys in `questions` become keys in `result["answers"]`. Use the returned probabilities when deciding whether to automate an action or ask a person to review it.

In [ ]:
model = CloudflareWorkersAIDecisionModel(
    model_name="@cf/cloudflare/clef-flash",
    account_id=account_id,
    api_token=api_token,
)
questions = {
    "urgent": {"type": "noul", "instructions": "Is this support request urgent?"},
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this request?",
        "criteria": {
            "billing": "Payments and invoices",
            "technical": "Outages and errors",
            "sales": "Plans and upgrades",
        },
    },
    "severity": {
        "type": "score",
        "instructions": "How severe is the customer impact?",
        "criteria": ["No impact", "Minor", "Major", "Critical"],
    },
}
result = model.evaluate(
    state="Checkout has been failing for every customer for the last hour.",
    questions=questions,
)
print(json.dumps(result["answers"], indent=2))

## Vision decision

Clef accepts embedded PNG, JPEG, and WebP images. This example passes a 16×16 red PNG as an image object; remote image URLs are not accepted. The questions ask about the image itself, and the returned answer includes probabilities for each option.

In [ ]:
red_png_base64 = (
    "iVBORw0KGgoAAAANSUhEUgAAABAAAAAQCAIAAACQkWg2AAAAF0lEQVR4nGP4z8BA"
    "EiJN9aiGUQ1DSgMAkPn/Afnh+ngAAAAASUVORK5CYII="
)
vision_model = CloudflareWorkersAIDecisionModel(
    model_name="@cf/cloudflare/clef",
    account_id=account_id,
    api_token=api_token,
)
vision_questions = {
    "red": {"type": "noul", "instructions": "Is the attached image predominantly red?"},
    "color": {
        "type": "choice",
        "instructions": "What is the dominant color of the attached image?",
        "criteria": {"red": "Red", "green": "Green", "blue": "Blue"},
    },
}
vision_result = vision_model.evaluate(
    state="Evaluate the attached image.",
    questions=vision_questions,
    images=[{"content_type": "image/png", "base64": red_png_base64}],
)
print(json.dumps(vision_result["answers"], indent=2))

## Python Worker binding

Inside a Python Worker, the existing AI binding supplies authentication. Use `aevaluate` there; Jupyter does not provide `env.AI`.

```python
model = CloudflareWorkersAIDecisionModel(binding=self.env.AI)
result = await model.aevaluate(
    state="Checkout is down.",
    questions={"urgent": {"type": "noul", "instructions": "Is this urgent?"}},
)
print(result["answers"])
```